# Práctica 1

Manuel Suano Gallardo

## Ejercicio 1

### E 1.1.

Primero, la estructura de la carpeta en local:

In [1]:
!ls

1_1_mapper.py	TP1_2024-25.pdf  convocatorias-2020.csv  practica1.ipynb
1_1_reducer.py	clients.csv	 countries.csv		 salida-02


Comenzaremos subiendo el archivo de datos `countries.csv` y `clientes.csv`

In [2]:
!hdfs dfs -mkdir /practica1
!hdfs dfs -mkdir /practica1/ejercicio1
!hdfs dfs -put /media/notebooks/Practica1/countries.csv /practica1/ejercicio1
!hdfs dfs -put /media/notebooks/Practica1/clients.csv /practica1/ejercicio1
!hdfs dfs -ls /practica1/ejercicio1

Found 2 items
-rw-r--r--   3 root supergroup       1289 2025-11-15 21:03 /practica1/ejercicio1/clients.csv
-rw-r--r--   3 root supergroup       4120 2025-11-15 21:03 /practica1/ejercicio1/countries.csv


Primero comprobramos la primera fila de cada archivo, por si pudieran llevar el nombre de las columnas

In [3]:
with open('./clients.csv', "r") as fp:
    print("Primera línea clients.csv: " + fp.readline())

with open('./countries.csv', "r") as fp:
    print("Primera línea countries.csv: " + fp.readline())

Primera línea clients.csv: Bertram Pearcy  ,bueno,SO

Primera línea countries.csv: Name,Code



Se puede observar que el archivo clients.csv no tiene el nombre de las columnas, mientras que countries.csv sí. Pasamos a ver un poco el contenido.

In [4]:
import pandas as pd

doc_countries = pd.read_csv('./countries.csv', header=0)

doc_countries.head()

,Name,Code
0,Afghanistan,AF
1,Åland Islands,AX
2,Albania,AL
3,Algeria,DZ
4,American Samoa,AS


In [5]:
doc_clients = pd.read_csv('./clients.csv', names=['Cliente','Opinion','Pais'])

doc_clients.head()

,Cliente,Opinion,Pais
0,Bertram Pearcy,bueno,SO
1,Steven Ulman,regular,ZA
2,Enid Follansbee,malo,GS
3,Candie Jacko,malo,SS
4,Alana Zufelt,regular,ES


A partir de esta información podemos constuir el mapper. Este recibirá los fichero y mostrará por la stdout, tanto los códigos de los países de cada valoración "bueno", como los pares de equivalencia entre el código del país y el nombre de este. Hay que tener en cuenta que la cabecera del archivo `countries.csv` también se recibirá pero no hay que enviarla.

```
mapper:
    equiv_cod_paises <- {}
    paises_bueno <- []

    para cada linea de stdin hacer
        div_linea <- dividir(linea, ',')

        si longitud(div_linea) = 3 y div_linea[1] es "bueno" entonces
            imprimir(div_linea[2])
        si longitud(div_linea) = 2 y div_linea[0] no es "Name" entonces
            imprimir(div_linea[0], div_linea[1])
```


In [6]:
%%writefile 1_1_mapper.py
#!/usr/bin/env python3

import sys

equiv_cod_paises = {}
paises_bueno = []

for linea in sys.stdin:
    # Dividimos la línea
    div_linea = linea.strip().split(',')

    # Es del fichero de opiniones y tiene por opinión "bueno"
    if len(div_linea) == 3 and div_linea[1] == 'bueno':
        # Se añade el país
        print(div_linea[2])
        
    # Es del fichero de países y no es la primera línea (el nombre de las columnas)
    elif len(div_linea) == 2 and div_linea[0] != 'Name':
        # Se manda la equivalencia entre ficheros
        print(f"{div_linea[0]},{div_linea[1]}")

Writing 1_1_mapper.py


El reducer irá realizando, por un lado, el conteo de los códigos, mientras que por otro se almacenarán las equivalencias para que cuando se haya terminado de recibir la información se pueda sustituir el código del país por el nombre, que quedará asociado al conteo realizado. Para mostrarlo por la stdout se ordenará los países alfabéticamente.

```
    reducer:
        conteo_valoraciones_paises <- {}
        equivalencia_paises <- {}

        para cada linea de stdin hacer
            si linea contiene ',' entonces
                nombre_pais, codigo_pais <- dividir(linea, ',')
                equivalencia_paises[codigo_pais] <- nombre_pais
            sino entonces
                # la línea es un código de país
                si linea no está en conteo_valoraciones_paises entonces
                    conteo_valoraciones_paises[linea] <- 0
                conteo_valoraciones_paises[linea] <- conteo_valoraciones_paises[linea] + 1

        conteo_valoraciones_paises <- renombrar_claves(conteo_valoraciones_paises, equivalencia_paises)

        para cada pais en ordenar(conteo_valoraciones_paises) hacer
            imprimir(pais, conteo_valoraciones_paises[pais])
```

In [48]:
%%writefile 1_1_reducer.py
#!/usr/bin/env python3

import sys

# Conteo de países
conteo_valoraciones_paises = {}
equivalencia_paises = {}

for linea in sys.stdin:
    linea = linea.strip()

    # Se trata de una equivalencia entre países
    if ',' in linea:
        linea = linea.split(',')
        nombre_pais = linea[0]
        cod_pais = linea[1]

        equivalencia_paises[cod_pais] = nombre_pais
    
    # Se trata de un código de un país con una valoración "bueno"
    else:
        if linea not in conteo_valoraciones_paises:
            conteo_valoraciones_paises[linea] = 0
        conteo_valoraciones_paises[linea] += 1

# Sustituimos las claves por el nombre de los paises
conteo_valoraciones_paises = { 
    equivalencia_paises[cod_pais] : conteo for cod_pais, conteo in conteo_valoraciones_paises.items() 
}

# Los mostramos ordenados alfabéticamente
for pais in sorted(conteo_valoraciones_paises):
    print(f'["{pais}"]\t{conteo_valoraciones_paises[pais]}')


Overwriting 1_1_reducer.py


Una vez se tiene el mapper y el reducer, realizamos la ejecución de Hadoop Streaming.

In [ ]:
!hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files 1_1_mapper.py,1_1_reducer.py \
-mapper 1_1_mapper.py \
-reducer 1_1_reducer.py \
-input /practica1/ejercicio1/clients.csv \
-input /practica1/ejercicio1/countries.csv \
-output /practica1/ejercicio1/salida_1_1

Visualizamos la salida del MapReduce

In [9]:
!hdfs dfs -cat /practica1/ejercicio1/salida_1_1/part-00000

["Canada"]	1
["Guam"]	3
["Guinea"]	1
["Portugal"]	1
["Qatar"]	1
["Somalia"]	1
["South Africa"]	1
["South Georgia and the South Sandwich Islands"]	1
["South Sudan"]	1
["Spain"]	3
["Turkey"]	1
["United States"]	1


### E 1.2.

Para obtener el país con mayor número de valoraciones "bueno" simplemente tendremos que recorrer el diccionario buscando el máximo. Al finalizar el bucle lo mostraremos por la stdout junto con el conteo asociado. El mapper se mantendrá el mismo.

```
    reducer:
        conteo_valoraciones_paises <- {}
        equivalencia_paises <- {}

        para cada linea de stdin hacer
            si linea contiene ',' entonces
                nombre_pais, codigo_pais <- dividir(linea, ',')
                equivalencia_paises[codigo_pais] <- nombre_pais
            sino entonces
                # la línea es un código de país
                si linea no está en conteo_valoraciones_paises entonces
                    conteo_valoraciones_paises[linea] <- 0
                conteo_valoraciones_paises[linea] <- conteo_valoraciones_paises[linea] + 1

        conteo_valoraciones_paises <- renombrar_claves(conteo_valoraciones_paises, equivalencia_paises)

        maximo <- -1
        pais_maximo <- Null
        para cada pais en conteo_valoraciones_paises hacer
            si conteo_valoraciones_paises[pais] > maximo entonces
                maximo <- conteo_valoraciones_paises[pais]
                pais_maximo <- pais

        imprimir(maximo, pais_maximo)
```

In [60]:
%%writefile 1_2_reducer.py
#!/usr/bin/env python3

import sys

# Conteo de países
conteo_valoraciones_paises = {}
equivalencia_paises = {}

for linea in sys.stdin:
    linea = linea.strip()

    # Se trata de una equivalencia entre países
    if ',' in linea:
        linea = linea.split(',')
        nombre_pais = linea[0]
        cod_pais = linea[1]

        equivalencia_paises[cod_pais] = nombre_pais
    
    # Se trata de un código de un país con una valoración "bueno"
    else:
        if linea not in conteo_valoraciones_paises:
            conteo_valoraciones_paises[linea] = 0
        conteo_valoraciones_paises[linea] += 1

# Sustituimos las claves por el nombre de los paises
conteo_valoraciones_paises = { 
    equivalencia_paises[cod_pais] : conteo for cod_pais, conteo in conteo_valoraciones_paises.items() 
}

# Recorremos buscando el país con mayor número de valoraciones
maximo = -1
pais_maximo = None
# Quitamos el orden alfabético por país
for pais in conteo_valoraciones_paises:
    if conteo_valoraciones_paises[pais] > maximo:
        maximo = conteo_valoraciones_paises[pais]
        pais_maximo = pais
    
print(f'{maximo}\t["{pais_maximo}"]')

Overwriting 1_2_reducer.py


In [62]:
!hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files 1_1_mapper.py,1_2_reducer.py \
-mapper 1_1_mapper.py \
-reducer 1_2_reducer.py \
-input /practica1/ejercicio1/clients.csv \
-input /practica1/ejercicio1/countries.csv \
-output /practica1/ejercicio1/salida_1_2

packageJobJar: [/tmp/hadoop-unjar1688439390115541363/] [] /tmp/streamjob8927471139713643079.jar tmpDir=null
2025-11-15 22:13:36,053 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2025-11-15 22:13:36,194 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2025-11-15 22:13:36,379 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1763222702967_0020
2025-11-15 22:13:36,735 INFO mapred.FileInputFormat: Total input files to process : 2
2025-11-15 22:13:36,840 INFO mapreduce.JobSubmitter: number of splits:3
2025-11-15 22:13:36,968 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1763222702967_0020
2025-11-15 22:13:36,968 INFO mapreduce.JobSubmitter: Executing with tokens: []
2025-11-15 22:13:37,162 INFO conf.Configuration: resource-types.xml not found
2025-11-15 22:13:37,162 INFO resource.ResourceUtils

Visualizamos la salida del MapReduce

In [63]:
!hdfs dfs -cat /practica1/ejercicio1/salida_1_2/part-00000

3	["Spain"]


### E 1.3.

Para este caso, en el bucle de mapper donde se busca el máximo, en lugar de guardar únicamente aquel que supera al máximo, en caso de encontrar otro país que igual lo que vamos a hacer es almacenarlo. Si se encuentra alguno que supere al máximo entonces tendremos que borrar la lista y empezar una nueva con el nuevo valor. De nuevo el mapper será el mismo.

```
    reducer:
        conteo_valoraciones_paises <- {}
        equivalencia_paises <- {}

        para cada linea de stdin hacer
            si linea contiene ',' entonces
                nombre_pais, codigo_pais <- dividir(linea, ',')
                equivalencia_paises[codigo_pais] <- nombre_pais
            sino entonces
                # la línea es un código de país
                si linea no está en conteo_valoraciones_paises entonces
                    conteo_valoraciones_paises[linea] <- 0
                conteo_valoraciones_paises[linea] <- conteo_valoraciones_paises[linea] + 1

        conteo_valoraciones_paises <- renombrar_claves(conteo_valoraciones_paises, equivalencia_paises)

        maximo <- -1
        pais_maximo <- []
        para cada pais en conteo_valoraciones_paises hacer
            si conteo_valoraciones_paises[pais] > maximo entonces
                maximo <- conteo_valoraciones_paises[pais]
                pais_maximo <- [pais]
            si conteo_valoraciones_paises[pais] = maximo entonces
                pais_maximo <- añadir(pais_maximo, pais)

        para cada pais en ordenado(pais_maximo) hacer
            imprimir(maximo, pais)
```

In [74]:
%%writefile 1_3_reducer.py
#!/usr/bin/env python3

import sys

# Conteo de países
conteo_valoraciones_paises = {}
equivalencia_paises = {}

for linea in sys.stdin:
    linea = linea.strip()

    # Se trata de una equivalencia entre países
    if ',' in linea:
        linea = linea.split(',')
        nombre_pais = linea[0]
        cod_pais = linea[1]

        equivalencia_paises[cod_pais] = nombre_pais
    
    # Se trata de un código de un país con una valoración "bueno"
    else:
        if linea not in conteo_valoraciones_paises:
            conteo_valoraciones_paises[linea] = 0
        conteo_valoraciones_paises[linea] += 1

# Sustituimos las claves por el nombre de los paises
conteo_valoraciones_paises = { 
    equivalencia_paises[cod_pais] : conteo for cod_pais, conteo in conteo_valoraciones_paises.items() 
}

# Recorremos buscando el país con mayor número de valoraciones
maximo = -1
paises_maximo = []
# Quitamos el orden alfabético por país
for pais in conteo_valoraciones_paises:
    if conteo_valoraciones_paises[pais] > maximo:
        maximo = conteo_valoraciones_paises[pais]
        paises_maximo = [pais]
    # Si es el mismo valor, se almacena
    elif conteo_valoraciones_paises[pais] == maximo:
        paises_maximo.append(pais)

# Ahora simplemente recorremos los países ordenados alfabéticamente
for pais_maximo in sorted(paises_maximo):
    print(f'{maximo}\t["{pais_maximo}"]')

Overwriting 1_3_reducer.py


In [78]:
!hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files 1_1_mapper.py,1_3_reducer.py \
-mapper 1_1_mapper.py \
-reducer 1_3_reducer.py \
-input /practica1/ejercicio1/clients.csv \
-input /practica1/ejercicio1/countries.csv \
-output /practica1/ejercicio1/salida_1_3

packageJobJar: [/tmp/hadoop-unjar3142157789730139195/] [] /tmp/streamjob7733212348678289346.jar tmpDir=null
2025-11-15 22:26:58,924 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2025-11-15 22:26:59,092 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2025-11-15 22:26:59,303 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1763222702967_0023
2025-11-15 22:26:59,757 INFO mapred.FileInputFormat: Total input files to process : 2
2025-11-15 22:26:59,881 INFO mapreduce.JobSubmitter: number of splits:3
2025-11-15 22:27:00,038 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1763222702967_0023
2025-11-15 22:27:00,039 INFO mapreduce.JobSubmitter: Executing with tokens: []
2025-11-15 22:27:00,246 INFO conf.Configuration: resource-types.xml not found
2025-11-15 22:27:00,247 INFO resource.ResourceUtils

In [79]:
!hdfs dfs -cat /practica1/ejercicio1/salida_1_3/part-00000

3	["Guam"]
3	["Spain"]


## Ejercicio 2